Notebook to 

In [ ]:
from census import Census
import pandas as pd
import os

In [131]:
api_key=os.environ.get('CENSUS_API_KEY')

if not api_key:
    raise ValueError("CENSUS_API_KEY environment variable is not set.")

census_api = Census(api_key, year = 2020)


In [ ]:
def get_census_race_data(census_api, year=2020):
    """
    Fetches and processes 2020 Census data for specific race groups.
    
    Parameters:
    - census_api: Your authenticated census API client instance.
    - table_type (str): 'P1' for total population, 'P2' for Hispanic/Latino population.
    - year (int): The census year (default 2020).
    """
    # 1. Define variable mappings based on table type

    vars_map = {
        "total": "P1_001N",
        "white_alone": "P1_003N",
        "sor_alone": "P1_008N",
        "white_and_sor": "P1_015N"
    }

    # 2. Build variables tuple for API call
    variables_of_interest = (
        vars_map["white_alone"], 
        vars_map["white_and_sor"], 
        vars_map["sor_alone"], 
        vars_map["total"], 
        "NAME"
    )

    # 3. Fetch data from Census API
    rows = census_api.pl.get(
        variables_of_interest,
        {"for": "county:*", "in": "state:*"},
        year=year
    )

    # 4. Create DataFrame and construct GEOID
    df = pd.DataFrame(rows)
    df["GEOID"] = df["state"] + df["county"]

    # 5. Convert relevant columns to numeric
    api_codes = [vars_map["total"], vars_map["white_alone"], vars_map["sor_alone"], vars_map["white_and_sor"]]
    df[api_codes] = df[api_codes].apply(pd.to_numeric)

    # 6. Filter down to columns we want and rename them dynamically
    df = df[["GEOID", "NAME"] + api_codes]
    df = df.rename(columns={
        vars_map["total"]: "total_population",
        vars_map["white_alone"]: "white_alone",
        vars_map["sor_alone"]: "sor_alone",
        vars_map["white_and_sor"]: "white_and_sor",
        "NAME": "county_name"
    })

    # 7. Calculate "other" category
    df["other"] = df["total_population"] - (df["white_alone"] + df["white_and_sor"] + df["sor_alone"])

    # 8. Calculate percentages for each group
    for col in ["white_alone", "sor_alone", "white_and_sor", "other"]:
        # Using a small epsilon or filling NaN handles cases where total population is 0
        df[f"{col}_pct"] = (df[col] / df["total_population"]) * 100

    return df

In [133]:
total_population_df = get_census_race_data(census_api, table_type="P1")

In [ ]:
import os
import re
import pandas as pd
import plotly.express as px
from typing import Literal
import numpy as np

def _slugify(text: str) -> str:  # filesystem-safe folder/file names
    text = text.strip().lower()
    text = re.sub(r"[^\w]+", "_", text) 
    return text.strip("_") or "unnamed"


def _resolve_categories(categories, type):
    if categories is None and type == "P1":
        return ["white_alone", "sor_alone", "white_and_sor", "other"]
    elif categories is None and type == "P2":
        return ["hispanic_white_alone", "hispanic_sor_alone", "hispanic_white_and_sor", "hispanic_other"]
    return categories


def _filter_state(df, state_fips):
    # 1. Isolate counties for ONLY the requested state using the GEOID prefix
    if state_fips:
        return df[df["GEOID"].str.startswith(state_fips)].copy()
    # If no state_fips is provided, use the entire DataFrame
    return df.copy()


def _weighted_averages(df_state, categories, denom_col):
    # 2. Calculate state-level averages for each category
    total_denom = df_state[denom_col].sum()
    averages = {col: (df_state[col].sum() / total_denom) * 100 for col in categories}
    return averages, total_denom


def _sort_by_white(df_state, type):
    # 3. Sort by 'white_alone' descending (Highest on left, lowest on right)
    key = "white_alone_pct" if type == "P1" else "hispanic_white_alone_pct"
    return df_state.sort_values(by=key, ascending=False)


def _build_area_chart(df_state, categories, place, group, type):
    custom_colors = ["#1d4e6f", "#2a9d8f", "#e9c46a", "#cbd2d9"]  # deep blue, teal, gold, cool grey

    # 4. Generate Interactive Chart
    fig = px.area(
        df_state,
        x="county_name",
        y=[c + "_pct" for c in categories],
        title=f"White and SOR Composition Across Counties in {place} ({group})",
        labels={"value": "Percentage (%)", "county_name": "County"},
        hover_name="county_name",  # Puts county name nicely at the very top of unified block
        color_discrete_sequence=custom_colors,
    )

    # 5. Handle formatting to prevent text duplication & fix crowded axes
    sort_label = "White Alone" if type == "P1" else "Hispanic White Alone"
    fig.update_layout(
        yaxis=dict(ticksuffix="%", range=[0, 100]),
        xaxis=dict(
            showticklabels=False,  # Clear county names from crowding bottom axis line (too crowded!)
            title="Counties (Sorted by {} Descending), Total Counties: {}".format(sort_label, len(df_state)),
            rangeslider=dict(visible=True),  # Keeps manual tracking simple if state has many counties
        ),
        legend_title_text="Race Group",
        hovermode="x unified",
    )
    # Cleans the individual row lines so they only show the percentage value
    fig.update_traces(hovertemplate="%{y:.1f}%")
    return fig


def _build_bar_chart(state_averages, categories, place, type):
    labels = [col.replace("_", " ").title() for col in categories]
    values = [state_averages[col] for col in categories]
    bar = px.bar(
        x=labels, y=values,
        title=f"Weighted {'Race' if type == 'P1' else 'Hispanic'} Profile in {place}",
        labels={"x": "Group", "y": "Share (%)"},
        text=[f"{v:.1f}%" for v in values],
    )
    bar.update_traces(textposition="outside")
    bar.update_layout(yaxis=dict(ticksuffix="%"), showlegend=False)
    return bar


def _build_report(df_state, categories, state_averages, place, total_denom, type):
    # Label the denominator by type: population for P1, Hispanic total for P2
    denom_label = "Total Population" if type == "P1" else "Total Hispanic Population"
    lines = []
    lines.append("Simple descriptive statistics for the counties in this state:")
    lines.append(df_state.describe().to_string())
    lines.append("\n" + "=" * 50)
    lines.append(f" STATEWIDE PROFILE CALCULATED AS WEIGHTED AVERAGE ({place.upper()}) ")
    lines.append(f" {denom_label}: {total_denom:,.0f}")
    lines.append("=" * 50)
    for col in categories:
        lines.append(f"{col.replace('_', ' ').title():<28} : {state_averages[col]:.2f}%")
    return "\n".join(lines)


def _save_outputs(out_dir, area_fig, bar_fig, report, df_state):
    # save everything into out_dir
    os.makedirs(out_dir, exist_ok=True)
    area_fig.write_html(os.path.join(out_dir, "composition_area.html"))
    bar_fig.write_html(os.path.join(out_dir, "weighted_profile_bar.html"))
    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        f.write(report + "\n")
    df_state.to_csv(os.path.join(out_dir, "county_data.csv"), index=False)

def _split_into_thirds(df_state):
    """Split an already-sorted frame into three equal-count groups.
    Assumes df_state is sorted by white_alone_pct descending.
    Returns list of (label, sub_df), highest-white third first."""
    index_chunks = np.array_split(df_state.index, 3)  # split the INDEX, not the frame
    labels = ["Highest White Alone Third", "Middle Third", "Lowest White Alone Third"]
    return [(label, df_state.loc[chunk]) for label, chunk in zip(labels, index_chunks)]


def plot_state_race_distributions(df: pd.DataFrame, state_fips: str, categories: list[str] = None,
                                  state_name: str = None, type: Literal["P1", "P2"] = "P1"):
    """
    Filters the census DataFrame for a single state, converts to percentages,
    sorts by White Alone descending, and generates a clean interactive area chart.

    Parameters:
    - df: The raw or filtered pandas DataFrame (must contain 'GEOID', 'county_name', and 'total_population').
    - state_fips: Two-character string matching the state code (e.g., "06" for California, "36" for NY). Input None to use the entire DataFrame.
    - categories: List of demographic columns to stack. Defaults to the 4 SOR groups.
    - state_name: Optional string for the state's full name, used in the chart title.
    - type: "P1" for total population, "P2" for Hispanic/Latino population. Determines which columns to use.
    """
    categories = _resolve_categories(categories, type)

    df_state = _filter_state(df, state_fips)
    if df_state.empty:
        print(f"Error: No data found for State FIPS '{state_fips}'")
        return

    denom_col = "total_population" if type == "P1" else "total_hispanic"
    state_averages, total_denom = _weighted_averages(df_state, categories, denom_col)
    df_state = _sort_by_white(df_state, type)

    group = "Hispanic Residents" if type == "P2" else "All Residents"
    place = state_name if state_name else f"State FIPS {state_fips}"
    out_dir = os.path.join(os.getenv("DATA_DIR", "."), f"{type}_{_slugify(place)}")

    area_fig = _build_area_chart(df_state, categories, place, group, type)
    area_fig.show()

    report = _build_report(df_state, categories, state_averages, place, total_denom, type)
    # print(report)

    bar_fig = _build_bar_chart(state_averages, categories, place, type)
    # bar_fig.show()

    # One area chart per equal-count third of counties
    for i, (third_label, sub_df) in enumerate(_split_into_thirds(df_state), start=1):
        place_third = f"{place} — {third_label}"
        area_fig = _build_area_chart(sub_df, categories, place_third, group, type)
        # area_fig.show()
        area_fig.write_html(os.path.join(out_dir, f"composition_area_third{i}.html"))
        sub_df.to_csv(os.path.join(out_dir, f"county_data_third{i}.csv"), index=False)

    _save_outputs(out_dir, area_fig, bar_fig, report, df_state)
    print(f"\nSaved outputs to: {out_dir}")
    return out_dir

In [188]:
# Plot all counties across all states, DC, and Puerto Rico
plot_state_race_distributions(total_population_df, state_fips=None, state_name="All 50 States + DC + Puerto Rico")


Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P1_all_50_states_dc_puerto_rico


'/Users/esher/mggg/projects/sor/sor_reboot/data/P1_all_50_states_dc_puerto_rico'

In [189]:
# Look at California counties only
plot_state_race_distributions(total_population_df, state_fips="06", state_name="California")

# Look at Texas counties only
plot_state_race_distributions(total_population_df, state_fips="48", state_name="Texas")


#Look at PR counties only
plot_state_race_distributions(total_population_df, state_fips="72", state_name="Puerto Rico")


Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P1_california



Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P1_texas



Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P1_puerto_rico


'/Users/esher/mggg/projects/sor/sor_reboot/data/P1_puerto_rico'

In [ ]:
def get_hispanic_racial_demographics(
    state_fips: str, census_client: Census
) -> pd.DataFrame:
    """Fetches 2020 PL 94-171 county-level data and derives Hispanic-only counts
    for four groupings: White alone, SOR alone, White and SOR, and all other
    responses combined. The four sum exactly to total_hispanic.
    """

    # Isolated cells: (Total from P1, Non-Hispanic from P2). Hispanic = P1 - P2.
    mappings = {
        "hispanic_white_alone": ("P1_003N", "P2_005N"),
        "hispanic_sor_alone": ("P1_008N", "P2_010N"),
        "hispanic_white_and_sor": ("P1_015N", "P2_017N"),
    }

    p1_vars = [pair[0] for pair in mappings.values()]
    p2_vars = [pair[1] for pair in mappings.values()]
    core_vars = ["NAME", "P1_001N", "P2_002N"]
    fields = tuple(core_vars + p1_vars + p2_vars)

    if not state_fips:
        state_fips = "*"
    
    raw_data = census_client.pl.get(
        fields=fields,
        geo={"for": "county:*", "in": f"state:{state_fips}"},
        year=2020,
    )

    df = pd.DataFrame(raw_data)
    numeric_cols = [col for col in df.columns if col not in ["NAME", "state", "county"]]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

    output_df = pd.DataFrame()
    output_df["county_name"] = df["NAME"]
    output_df["state_fips"] = df["state"]
    output_df["county_fips"] = df["county"]
    output_df["total_population"] = df["P1_001N"]
    output_df["total_hispanic"] = df["P2_002N"]
    output_df["GEOID"] = df["state"] + df["county"]

    # Three isolated cells
    for label, (p1_col, p2_col) in mappings.items():
        output_df[label] = df[p1_col] - df[p2_col]

    # Fourth grouping: everything else, as a residual off total_hispanic
    output_df["hispanic_other"] = output_df["total_hispanic"] - (
        output_df["hispanic_white_alone"]
        + output_df["hispanic_sor_alone"]
        + output_df["hispanic_white_and_sor"]
    )

    # The three cells are a subset of total_hispanic, so each and the residual are non-negative.
    derived = ["hispanic_white_alone", "hispanic_sor_alone", "hispanic_white_and_sor", "hispanic_other"]
    assert (output_df[derived] >= 0).all().all(), "Negative derived count; check P1/P2 mapping."

    denom = output_df["total_hispanic"].replace(0, pd.NA)
    for col in derived:
        output_df[f"{col}_pct"] = (output_df[col] / denom) * 100


    return output_df

hispanic_latino_population_df = get_hispanic_racial_demographics(state_fips=None, census_client=census_api)



In [191]:


print("Zero-Hispanic counties:", (hispanic_latino_population_df["total_hispanic"] == 0).sum())
print("Minimum total_hispanic:", hispanic_latino_population_df["total_hispanic"].min())

floor = hispanic_latino_population_df["total_hispanic"].min()
print(hispanic_latino_population_df[hispanic_latino_population_df["total_hispanic"] == floor][["county_name", "total_hispanic"]])

Zero-Hispanic counties: 0
Minimum total_hispanic: 1.0
               county_name  total_hispanic
2489  Loving County, Texas             1.0


In [192]:
# Plot all counties across all states, DC, and Puerto Rico
plot_state_race_distributions(hispanic_latino_population_df, state_fips=None, state_name="All 50 States + DC + Puerto Rico", type="P2")

#Look at California counties only
plot_state_race_distributions(hispanic_latino_population_df, state_fips="06", state_name="California", type="P2")

# Look at Texas counties only
plot_state_race_distributions(hispanic_latino_population_df, state_fips="48", state_name="Texas", type="P2")

#Look at PR counties only
plot_state_race_distributions(hispanic_latino_population_df, state_fips="72", state_name="Puerto Rico", type="P2")


Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P2_all_50_states_dc_puerto_rico



Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P2_california



Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P2_texas



Saved outputs to: /Users/esher/mggg/projects/sor/sor_reboot/data/P2_puerto_rico


'/Users/esher/mggg/projects/sor/sor_reboot/data/P2_puerto_rico'